# TinyFace alignment benchmark — local Jupyter
Chạy trực tiếp trong kernel Jupyter/GPU local. Không dùng Google Colab API hoặc Modal.

In [ ]:
# Bootstrap: đúng repository và đúng Python kernel
from pathlib import Path
import os,sys,subprocess,importlib,importlib.util
ROOT=Path.cwd()
if not (ROOT/'CVLface-main').is_dir():
 candidates=[Path('/content/Landmark-aligmenttest'),*ROOT.parents]
 ROOT=next((p for p in candidates if (p/'CVLface-main').is_dir()),None)
if ROOT is None:raise FileNotFoundError('Không tìm thấy repository chứa CVLface-main')
REPO=ROOT/'CVLface-main';SRC=REPO/'cvlface';DATA=REPO/'tinyface';RESULTS=ROOT/'results';CACHE=ROOT/'model_cache'
utils_file=SRC/'general_utils'/'huggingface_model_utils.py'
if not utils_file.is_file():raise FileNotFoundError(f'Thiếu source local: {utils_file}')
for p in (DATA,RESULTS,CACHE):p.mkdir(parents=True,exist_ok=True)
if str(SRC) not in sys.path:sys.path.insert(0,str(SRC))
def pip_install(*specs):
 print('Installing into',sys.executable,':',*specs)
 subprocess.check_call([sys.executable,'-m','pip','install','-q',*specs])
 importlib.invalidate_caches()
def ensure_import(module,package=None):
 try:return importlib.import_module(module)
 except Exception as first:
  pip_install(package or module)
  try:return importlib.import_module(module)
  except Exception as second:raise RuntimeError(f'Không import được {module} bằng {sys.executable} sau khi cài {package or module}: {second}') from first
print('ROOT=',ROOT);print('Python=',sys.executable)


In [ ]:
# Dependency riêng cho tải dữ liệu; cell này có thể chạy độc lập sau bootstrap
gdown=ensure_import('gdown','gdown')
print('gdown OK:',gdown.__version__)


In [ ]:
# Download/unzip TinyFace
import zipfile,tempfile,shutil
FILE_ID='1xTZc7lNmWN33ECO2AKH6FycGdiqIK7W0';ZIP=ROOT/'tinyface.zip'
if not (DATA/'Testing_Set/Probe').is_dir():
 out=gdown.download(id=FILE_ID,output=str(ZIP),quiet=False)
 if not out or not ZIP.is_file():raise RuntimeError('Không tải được ZIP; Drive phải bật Anyone with the link')
 with tempfile.TemporaryDirectory() as td:
  with zipfile.ZipFile(ZIP) as z:z.extractall(td)
  src=next((p for p in [Path(td),*Path(td).rglob('*')] if p.is_dir() and (p/'Testing_Set/Probe').is_dir()),None)
  if src is None:raise RuntimeError('ZIP không có Testing_Set/Probe')
  if DATA.exists():shutil.rmtree(DATA)
  shutil.copytree(src,DATA)
for s in ('Probe','Gallery_Match','Gallery_Distractor'):
 p=DATA/'Testing_Set'/s
 if not p.is_dir():raise FileNotFoundError(p)
 print(s,len(list(p.glob('*.jpg'))))


In [ ]:
# Model dependencies: kiểm tra trước, chỉ cài package thiếu/lỗi
requirements=[('numpy','numpy==1.26.4'),('torch','torch'),('torchvision','torchvision'),('transformers','transformers==4.34.1'),('huggingface_hub','huggingface-hub'),('safetensors','safetensors'),('omegaconf','omegaconf'),('cv2','opencv-python-headless'),('onnx','onnx'),('mediapipe','mediapipe==0.10.14'),('pandas','pandas'),('PIL','pillow'),('tqdm','tqdm'),('skimage','scikit-image'),('scipy','scipy'),('tabulate','tabulate'),('matplotlib','matplotlib'),('timm','timm==0.9.7'),('easydict','easydict==1.10'),('pyrootutils','pyrootutils==1.0.4'),('einops','einops==0.6.1'),('packaging','packaging')]
loaded={}
for module,spec in requirements:loaded[module]=ensure_import(module,spec)
# ORT cần bản CUDA 12; gỡ bản CUDA 13 nếu import lỗi
try:
 import torch
 ort=importlib.import_module('onnxruntime')
except Exception as exc:
 print('Replacing incompatible ONNX Runtime:',exc)
 subprocess.run([sys.executable,'-m','pip','uninstall','-y','onnxruntime','onnxruntime-gpu'],check=False,stdout=subprocess.DEVNULL)
 pip_install('onnxruntime-gpu==1.20.1')
 ort=importlib.import_module('onnxruntime')
insightface=ensure_import('insightface','insightface==0.7.3')
from general_utils.huggingface_model_utils import load_model_from_local_path
print('All model dependencies and CVLFace imports OK')


In [ ]:
# Environment report
import platform,importlib.metadata as md,torch
print('Python executable:',sys.executable)
print('Python version:',platform.python_version())
subprocess.run([sys.executable,'-m','pip','--version'],check=True)
print('Torch:',torch.__version__,'CUDA:',torch.version.cuda,'CUDA available:',torch.cuda.is_available())
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('ORT:',ort.__version__,'providers:',ort.get_available_providers())
for package in ['gdown','transformers','huggingface-hub','onnxruntime-gpu','insightface','mediapipe','timm']:
 try:print(package,md.version(package))
 except md.PackageNotFoundError:print(package,'NOT INSTALLED')


In [ ]:
# Load recognizer and four alignment pipelines
import json,time,cv2,numpy as np,torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from huggingface_hub import snapshot_download
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type!='cuda':raise RuntimeError('GPU CUDA is required for full TinyFace')
def model(repo):
 p=CACHE/repo.replace('/','__')
 if not (p/'config.json').exists():snapshot_download(repo_id=repo,local_dir=str(p),token=os.getenv('HF_TOKEN'))
 return load_model_from_local_path(str(p)).to(DEVICE).eval()
recognizer=model('minchul/cvlface_adaface_vit_base_kprpe_webface4m')
dfa={'dfa_mobilenet':model('minchul/cvlface_DFA_mobilenet'),'dfa_resnet50':model('minchul/cvlface_DFA_resnet50')}
from insightface.app import FaceAnalysis
providers=['CUDAExecutionProvider','CPUExecutionProvider'] if 'CUDAExecutionProvider' in ort.get_available_providers() else ['CPUExecutionProvider']
scrfd=FaceAnalysis(name='buffalo_l',root=str(CACHE/'insightface'),allowed_modules=['detection'],providers=providers);scrfd.prepare(ctx_id=0 if providers[0]=='CUDAExecutionProvider' else -1,det_thresh=.3,det_size=(640,640))
import mediapipe as mp
mesh=mp.solutions.face_mesh.FaceMesh(static_image_mode=True,max_num_faces=1,refine_landmarks=False,min_detection_confidence=.3)
ARC=np.array([[38.2946,51.6963],[73.5318,51.5014],[56.0252,71.7366],[41.5493,92.3655],[70.7299,92.2041]],np.float32)
def square(p):
 x=np.asarray(Image.open(p).convert('RGB'));h,w=x.shape[:2];s=max(h,w);y=np.zeros((s,s,3),np.uint8);y[(s-h)//2:(s-h)//2+h,(s-w)//2:(s-w)//2+w]=x;return y
def ten(x):return torch.from_numpy(x.copy()).permute(2,0,1).float().div(127.5).sub(1).unsqueeze(0).to(DEVICE)
def warp(x,k):
 M,_=cv2.estimateAffinePartial2D(k.astype(np.float32),ARC,method=cv2.LMEDS)
 if M is None:raise RuntimeError('transform failed')
 y=cv2.warpAffine(x,M,(112,112));ak=np.c_[k,np.ones(5)]@M.T
 return ten(y),torch.tensor(ak/112,dtype=torch.float32,device=DEVICE)[None]
def infer(path,name):
 x=square(path)
 if name in dfa:
  o=dfa[name](ten(x));return o[0],o[2],float(o[3].item())
 if name=='scrfd10g':
  fs=scrfd.get(cv2.cvtColor(x,cv2.COLOR_RGB2BGR))
  if not fs:raise RuntimeError('no face')
  f=max(fs,key=lambda z:(z.bbox[2]-z.bbox[0])*(z.bbox[3]-z.bbox[1]));a,k=warp(x,np.asarray(f.kps,np.float32));return a,k,float(f.det_score)
 r=mesh.process(cv2.resize(x,(640,640),interpolation=cv2.INTER_CUBIC))
 if not r.multi_face_landmarks:raise RuntimeError('no face mesh')
 p=r.multi_face_landmarks[0].landmark
 def m(ii):return np.mean([[p[i].x*x.shape[1],p[i].y*x.shape[0]] for i in ii],0)
 a,k=warp(x,np.asarray([m([33,133]),m([362,263]),m([1]),m([61]),m([291])],np.float32));return a,k,1.
def embedding(a,k):return F.normalize(recognizer(a,k).float(),dim=1)[0].cpu().numpy().astype('float32')
print('Models loaded on',DEVICE,'SCRFD providers',providers)


In [ ]:
# Preflight then full run with resume every 100 images
splits={'probe':sorted((DATA/'Testing_Set/Probe').glob('*.jpg')),'gallery_match':sorted((DATA/'Testing_Set/Gallery_Match').glob('*.jpg')),'distractor':sorted((DATA/'Testing_Set/Gallery_Distractor').glob('*.jpg'))}
items=[(p,s) for s,ps in splits.items() for p in ps];PIPELINES=['dfa_mobilenet','dfa_resnet50','scrfd10g','mediapipe']
def label(p):return int(p.stem.split('_')[0])
for n in PIPELINES:
 with torch.inference_mode():a,k,c=infer(items[0][0],n);e=embedding(a,k)
 assert a.shape[-2:]==(112,112) and k.shape==(1,5,2) and e.shape==(512,);print(n,'preflight OK')
def run(name):
 cp=RESULTS/f'{name}.npz';rows=[];start=0
 if cp.exists():z=np.load(cp,allow_pickle=True);rows=z['rows'].tolist();start=int(z['next']);print('resume',name,start)
 with torch.inference_mode():
  for i,(p,s) in enumerate(tqdm(items[start:],desc=name),start):
   t=time.perf_counter();r={'path':str(p),'split':s,'label':label(p) if s!='distractor' else -100,'ok':False,'error':'','embedding':None}
   try:a,k,c=infer(p,name);r.update(ok=c>=.3,confidence=c,embedding=embedding(a,k))
   except Exception as ex:r['error']=f'{type(ex).__name__}: {ex}'
   r['latency_ms']=(time.perf_counter()-t)*1000;rows.append(r)
   if (i+1)%100==0 or i+1==len(items):np.savez_compressed(cp,rows=np.array(rows,dtype=object),next=i+1)
 return rows
results={n:run(n) for n in PIPELINES}


In [ ]:
# Metrics and reports
import pandas as pd
def metric(rows):
 good=[r for r in rows if r['ok'] and r['embedding'] is not None];by={r['path']:r['embedding'] for r in good};gal=[r for r in rows if r['split']!='probe' and r['path'] in by];G=np.stack([by[r['path']] for r in gal]);gl=np.array([r['label'] for r in gal]);ranks={1:[],5:[],20:[]}
 for p in [r for r in rows if r['split']=='probe']:
  hit=np.zeros(len(G),bool) if p['path'] not in by else gl[np.argsort(-(G@by[p['path']]))]==p['label']
  for q in ranks:ranks[q].append(float(hit[:q].any()))
 lat=[r['latency_ms'] for r in rows];return {'coverage':np.mean([r['ok'] for r in rows]),'rank1':100*np.mean(ranks[1]),'rank5':100*np.mean(ranks[5]),'rank20':100*np.mean(ranks[20]),'p50_ms':np.percentile(lat,50),'p95_ms':np.percentile(lat,95)}
summary=pd.DataFrame([{'pipeline':n,**metric(r)} for n,r in results.items()]);summary.to_csv(RESULTS/'summary.csv',index=False);(RESULTS/'results.json').write_text(json.dumps({n:metric(r) for n,r in results.items()},indent=2));(RESULTS/'benchmark_report.md').write_text('# TinyFace local benchmark\n\nSingle-view, no flip TTA.\n\n'+summary.to_markdown(index=False));display(summary);print(RESULTS)
